# Homework 2
## Анализ текста, текстовые модели


### Часть 2: Классификация тональности с помощью BERT

Задача:

Цель этой задачи — использовать предварительно обученную модель BERT для классификации тональности рецензий на фильмы.

[IMDB Dataset](https://disk.yandex.ru/d/dhKpEgM4rQkLiQ)

#### **Задание для части 2:**

Используйте предварительно обученную модель BERT для классификации тональности отзывов о фильмах. (Вес задания: 20%)

In [92]:
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader

from transformers import BertModel, BertPreTrainedModel, BertConfig
from transformers import BertTokenizer, BertForSequenceClassification

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

from tqdm import tqdm

import warnings
import gdown
import os

warnings.filterwarnings('ignore')

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"current device: {device}")


current device: mps


In [93]:
# download csv if there is not; 

if not os.path.exists('IMDB.csv'):
    url = 'https://docs.google.com/spreadsheets/d/e/2PACX-1vT6-n57dcXO2nHTwmAkZSw_MGYGSd5vFIE0TtrnmePnzM37Jev_0mpXdVbLZqVLcnAYkyBpkztKpQH2/pub?output=csv'
    gdown.download(url, 'IMDB.csv', quiet=False)


In [94]:
# loading data; splitting 

imdb_raw = pd.read_csv('IMDB.csv')
texts = imdb_raw['review'].values
labels = imdb_raw['sentiment'].map({'negative': 0, 'positive': 1}).values

train_texts, test_texts, train_labels, test_labels = train_test_split(
			texts, 
			labels, 
			test_size=0.2, 
			random_state=666, 
			stratify=labels
			)

train_texts = train_texts.tolist()
test_texts = test_texts.tolist()


In [95]:
# imdbert 

class IMDBert(BertPreTrainedModel):
    all_tied_weights_keys = dict()
    
    def __init__(self, config):
        super().__init__(config)
        # Основная BERT-модель (без головы)
        self.bert = BertModel(config)
        # Дропаут для регуляризации (можно взять из config.hidden_dropout_prob)
        self.dropout = nn.Dropout(config.hidden_dropout_prob)
        # Классификационная голова: от размера скрытого слоя (768) до числа меток (2)
        self.classifier = nn.Linear(config.hidden_size, config.num_labels)
        # Инициализация весов (важно для стабильности)
        self.init_weights()

    def forward(self, input_ids, attention_mask=None, token_type_ids=None, labels=None):
        # Прогоняем через BERT
        outputs = self.bert(
            input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids
        )
        # Берём пулерованный выход ([CLS] вектор после Linear+Tanh)
        pooled_output = outputs.pooler_output
        # Применяем дропаут
        pooled_output = self.dropout(pooled_output)
        # Получаем логиты для двух классов
        logits = self.classifier(pooled_output)

        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, self.config.num_labels), labels.view(-1))

        # Возвращаем tuple (loss, logits) — так принято в transformers
        return (loss, logits) if loss is not None else logits


In [96]:
class IMDbDataset(Dataset):
    def __init__(self, encodings):
        self.input_ids = encodings['input_ids']
        self.attention_mask = encodings['attention_mask']
        self.labels = encodings['labels']

    def __getitem__(self, idx):
        return {
            'input_ids': self.input_ids[idx],
            'attention_mask': self.attention_mask[idx],
            'labels': self.labels[idx]
        }

    def __len__(self):
        return len(self.labels)


In [97]:
config = BertConfig.from_pretrained('bert-base-uncased', num_labels=2)
model = IMDBert.from_pretrained('bert-base-uncased', config=config)
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased', do_lower_case=True)

model.to(device)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

IMDBert LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


IMDBert(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=Tr

In [98]:
MAX_LEN = 512
BATCH_SIZE = 12
EPOCHS = 4

In [99]:
def tokenize_texts(texts, labels, max_len=MAX_LEN):
    encodings = tokenizer(
        texts,
        truncation=True,
        padding=True,
        max_length=max_len,
        return_tensors=None
    )
    return {
        'input_ids': torch.tensor(encodings['input_ids']),
        'attention_mask': torch.tensor(encodings['attention_mask']),
        'labels': torch.tensor(labels, dtype=torch.long)
    }


In [100]:
train_encodings = tokenize_texts(train_texts, train_labels)
test_encodings = tokenize_texts(test_texts, test_labels)


In [101]:
train_dataset = IMDbDataset(train_encodings)
test_dataset = IMDbDataset(test_encodings)


In [102]:
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)


In [ ]:
batch = next(iter(train_loader))
print(batch['input_ids'].shape)       # (batch_size, max_len)
print(batch['attention_mask'].shape)  # (batch_size, max_len
print(batch['labels'].shape)          # (batch_size,)


torch.Size([12, 512])
torch.Size([12, 512])
torch.Size([12])


In [104]:
def train_epoch(model, dataloader, optimizer, device):
    model.train()
    total_loss = 0
    for batch in tqdm(dataloader, desc='Training'):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs[0] if isinstance(outputs, tuple) else outputs.loss
        total_loss += loss.item()

        loss.backward()
        optimizer.step()

    return total_loss / len(dataloader)

def evaluate(model, dataloader, device):
    model.eval()
    preds, true_labels = [], []
    with torch.no_grad():
        for batch in tqdm(dataloader, desc='Evaluating'):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids, attention_mask=attention_mask)
            logits = outputs[1] if isinstance(outputs, tuple) else outputs.logits
            preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
            true_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(true_labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(true_labels, preds, average='binary')
    return acc, precision, recall, f1


In [105]:
optimizer = AdamW(model.parameters(),
                  lr=2e-5,
                  eps=1e-7)

for epoch in range(EPOCHS):
    print(f"\n=== Epoch {epoch+1}/{EPOCHS} ===")
    train_loss = train_epoch(model, train_loader, optimizer, device)
    acc, precision, recall, f1 = evaluate(model, test_loader, device)
    print(f"Train loss: {train_loss:.4f}")
    print(f"Test accuracy: {acc:.4f}")
    print(f"Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {f1:.4f}")

    torch.save(model.state_dict(), f'imdbert_epoch{epoch+1}.pth')


=== Epoch 1/4 ===


Training:   0%|          | 5/3334 [00:36<6:44:53,  7.30s/it]


KeyboardInterrupt: 